In [ ]:
import sys, os, glob, shutil, subprocess
import torch

if not torch.cuda.is_available():
    raise SystemExit("GPU не выделена — проверьте ускоритель в настройках ноутбука")
print("GPU:", torch.cuda.get_device_name(0))

found = glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)
if not found:
    raise SystemExit("Пакет данных не найден в /kaggle/input")
pack = os.path.dirname(found[0])
folds = glob.glob("/kaggle/input/**/llm_train_f1.parquet", recursive=True)
if not folds:
    raise SystemExit("Файлы фолда 1 не найдены")
# Файлы фолда лежат в отдельном датасете — кладём рядом с пакетом, чтобы поиск их нашёл.
os.makedirs("/kaggle/working/pack", exist_ok=True)
for path in glob.glob(pack + "/*") + glob.glob(os.path.dirname(folds[0]) + "/*.parquet"):
    dst = "/kaggle/working/pack/" + os.path.basename(path)
    if not os.path.exists(dst):
        os.symlink(path, dst)
print("пакет:", sorted(os.listdir("/kaggle/working/pack")))

modules = glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
for path in glob.glob(os.path.dirname(modules[0]) + "/*.py"):
    shutil.copy(path, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()

subprocess.run([sys.executable, "-u", "-m", "src.train_ce_large",
                "--prepacked", "/kaggle/working/pack", "--holdout-fold", "1",
                "--epochs", "1", "--batch-size", "256", "--max-length", "192",
                "--output", "/kaggle/working/ce_fold1"],
               check=True, cwd="/kaggle/working",
               env=dict(os.environ, PYTHONPATH="/kaggle/working"))
